## Setting UP external libraries and Database

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df=pd.read_csv("../data/insurance.csv")

## Analysis

In [ ]:
df.info()

# Inference :
# 1. There are 7 columns with 1337 entries 
# 2. The Dataset is complete and there is no need of imputation

In [ ]:
df.columns = [x.upper() if x=="bmi" else x.title()  for x in df.columns]
df.columns

# Action:
# The column names are converted to title casing

In [ ]:
df["Children"].value_counts()

# Inference 
# People have 0 to 5 children

In [ ]:
col_num = list(df.select_dtypes(include = ["number"]).columns)
col_num.remove("Charges")
col_cat = list(df.select_dtypes(include = ["object", "category", "str"]).columns)
col_label = ["Charges"]
print(f"Numerical Columns : {col_num}")
print(f"Categorical Columns : {col_cat}")
print(f"Label Columns : {col_label}")

In [ ]:
sns.pairplot(df)

# Inference 
# 1. BMI distribution follows a bell shaped curve
# 2. The charges vs Age graph shows that there is a hidden linear pattern
# 3. The charges vs BMI graph shows that some people regardless of the BMI have high charges

In [ ]:
fig1, axs1 = plt.subplots(1,3, figsize=(14,4))

sns.boxplot(df, x="Sex", y="Charges", ax=axs1[0])
axs1[0].set_title("Charges vs Sex")
axs1[0].set_xlabel("Gender")

sns.boxplot(df, x="Smoker", y="Charges", ax=axs1[1])
axs1[1].set_title("Charges vs Smoker")
axs1[1].set_xlabel("Smoking Status")
axs1[1].set_ylabel("")

sns.boxplot(df, x="Region", y="Charges", ax=axs1[2])
axs1[2].set_title("Region vs Sex")
axs1[2].set_xlabel("Region")
axs1[2].set_ylabel("")

# Inference:
# There is no clear connection between Sex and Charges
# There is not clear connection between Region and Charges
# There is a cleat connection between Smoker and Charges

In [ ]:
g = sns.pairplot(df, x_vars=["Age", "BMI", "Children"], y_vars=["Charges"], hue="Smoker", palette = ["red", "blue"])
g.axes[0,0].set_title("Charges vs Age (strat-sex)", fontsize = 10)
g.axes[0,1].set_title("Charges vs BMI (strat-sex)", fontsize = 10)
g.axes[0,2].set_title("Charges vs Children (strat-sex)", fontsize = 10)

# Inference:
# 1. Charges shows a clear linear relationship with Age but there is also a factor of Smoking status
# 2. Charges shows a linear relationship with Age but there is also a factor of Smoking status

In [ ]:
fig2, axs2 = plt.subplots(1,2, figsize=(10,4))
axs2[0].boxplot(df["Charges"])
axs2[0].set_xticks([1], labels = [""])
axs2[0].set_ylabel("Charges")
sns.kdeplot(df["Charges"], fill=True, ax=axs2[1])

# Inference:
# Labels has a right-tail

### Output graphs

In [ ]:
# mkdir -p /kaggle/working/graphs

In [ ]:
# fig1.savefig("output_files/Charges_vs_cat.png")
# g.savefig("output_files/Charges_vs_num_smoker.png")
# fig2.savefig("output_files/Charges_distribution.png")

## New useful columns

In [ ]:
df["Age"].describe()

# Inference
# Maximum age = 64
# Minimum age = 18

In [ ]:
Age_Group = pd.Series(pd.cut(df["Age"],
                             bins = [18, 25, 39, 54, 64],
                             labels = ["young_adult", "reproductive_peak", "mid_life", "pre-medicare"]))
sns.barplot(Age_Group.value_counts())

In [ ]:
Strat_col = pd.Series(map(str, Age_Group)) + "_" +pd.Series(map(str, df["Sex"]))
print(Strat_col)

## Stratification and label segregation

In [ ]:
from sklearn.model_selection import train_test_split

data_training, data_testing = train_test_split(df, test_size=.2 , stratify = Strat_col, random_state=42)

# Action:
# Stratification done on the basis of age_group and gender

In [ ]:
print(len(data_training[data_training["Sex"]=="female"])/len(data_training))
print(len(data_testing[data_testing["Sex"]=="female"])/len(data_testing))
print(len(df[df["Sex"]=="female"])/len(df))

# Inference:
# Stratification done successfully

In [ ]:
data_training_copy = data_training.copy()
label = pd.Series(data_training_copy["Charges"])

# Action:
# 1. A copy of training set is created
# 2. Label is separated

## Features Transformation

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
import sys
sys.path.append("..")  # if running from ml/notebooks/

from transformers import (
    smokers_fun,
    smokers_feature_name,
    non_smokers_fun,
    non_smokers_feature_name,
)

pipeline_cat = make_pipeline(OneHotEncoder())
pipeline_num = make_pipeline(StandardScaler())

# def smokers_fun(x):
#     feature_series = x.iloc[:, 0]
#     smokers_series = x.iloc[:, 1]
#     return (feature_series*((smokers_series == "yes").astype(int))).to_frame()
#
# def smokers_feature_name(function_transformer, feature_names_in):
#     return [str(feature_names_in[0])+"_of_smokers"]

pipeline_integrated_smoking = make_pipeline(FunctionTransformer(smokers_fun, feature_names_out=smokers_feature_name), StandardScaler())

# def non_smokers_fun(x):
#     feature_series = x.iloc[:, 0]
#     non_smokers_series = x.iloc[:, 1]
#     return (feature_series*((non_smokers_series == "no").astype(int))).to_frame()
#
# def non_smokers_feature_name(function_transformer, feature_names_in):
#     return [str(feature_names_in[0])+"_of_non_smokers"]

pipeline_integrated_non_smoking = make_pipeline(FunctionTransformer(non_smokers_fun, feature_names_out=non_smokers_feature_name), StandardScaler())

# Actions:
# 1. Since charges depends linearly on BMI but the linear relation differs based on whether the person is smoker or not, Smokers' BMI & non-smokers' BMI are teated as different dimensions.
# 2. For same reason Smokers' Age & non-smokers' Age are teated as different dimensions.

In [ ]:
pipeline_num

In [ ]:
pipeline_cat

In [ ]:
pipeline_integrated_smoking

In [ ]:
pipeline_integrated_non_smoking

In [ ]:
from sklearn.compose import ColumnTransformer

preprocessing = ColumnTransformer([
    ("pipeline_num_default", pipeline_num, ["Children"]),
    ("pipeline_cat_default", pipeline_cat, ["Sex", "Region"]),
    ("BMI_smoking", pipeline_integrated_smoking, ["BMI", "Smoker"]),
    ("Age_smoking", pipeline_integrated_smoking, ["Age", "Smoker"]),
    ("BMI_non_smoking", pipeline_integrated_non_smoking, ["BMI", "Smoker"]),
    ("Age_non_smoking", pipeline_integrated_non_smoking, ["Age", "Smoker"])
])

# Actions:
# 1. Children column is subjected to default numerical pipeline
# 1. Sex and Region column are subjected to default categorical pipeline
# 1. BMI and Age columns are subjected to smokers and non-smokers pipeline


In [ ]:
preprocessing

## Engaging Model

In [ ]:
from sklearn.linear_model import LinearRegression

linear_model = make_pipeline(preprocessing, LinearRegression())

In [ ]:
linear_model.fit(data_training_copy, label)

### Output model

In [ ]:
import joblib
joblib.dump(linear_model, "../models/linear_model.pkl")

## Test on Training set and Cross-Validation

In [ ]:
predictions_training = linear_model.predict(data_training_copy)
print(predictions_training[:5].round(2))
print(label[:5].round(2).to_list())

In [ ]:
from sklearn.metrics import root_mean_squared_error
rmse_training = root_mean_squared_error(label, predictions_training)
print(f"rmse_training = {rmse_training}")

In [ ]:
from sklearn.model_selection import cross_val_score

rmse_train_10folds = -cross_val_score(linear_model, data_training_copy, label, scoring="neg_root_mean_squared_error", cv=10)
print(f"10-folds cross validation rmse :\n{pd.Series(rmse_train_10folds).describe()}")

## Predictions on Test set

In [ ]:
predictions_testing = linear_model.predict(data_testing)
print(predictions_testing[:5].round(2))
print(data_testing["Charges"][:5].round(2).to_list())

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score

rmse_testing = root_mean_squared_error(data_testing["Charges"], predictions_testing)
mae_testing = mean_absolute_error(data_testing["Charges"], predictions_testing)
r2_testing = r2_score(data_testing["Charges"], predictions_testing)

print(f"rmse_testing = {rmse_testing}")
print(f"mae_testing = {mae_testing}")
print(f"mae_testing/median_charges = { (mae_testing/np.median(data_testing["Charges"])*100).round(2) }%")
print(f"r2_testing = {r2_testing}")

### Output predictions

In [ ]:
# output = data_testing.copy()
# output["Predicted_Charges"] = predictions_testing
# output["Residual"] = output["Charges"] - output["Predicted_Charges"]
#
# output.to_csv("output_files/insurance_predictions.csv", index=False)

In [ ]:
from pathlib import Path
model_path = Path("../models/linear_model.pkl")

import joblib
model = joblib.load(model_path)

In [ ]:
input_data = pd.DataFrame({"Age":[29],
                           "Sex":["male"],
                           "BMI":[30],
                           "Smoker":["yes"],
                           "Children":[0],
                           "Region": ["southwest"]})
user_prediction = model.predict(input_data)
print(user_prediction)

In [ ]:
print(smokers_fun.__module__)

In [ ]:
import pickletools
with open("../models/linear_model.pkl", "rb") as f:
    pickletools.dis(f.read(2000))  # inspect first part of the pickle stream